# 02-12 nnU-Net v2 PHE-SICH PHE baseline

Notebook này dùng framework `nnUNet-master` đã tải local để tạo baseline nnU-Net v2 cho PHE segmentation.

Phạm vi bản này:

- Chỉ dùng `PHE-SICH-CT-IDS` vì đây là bộ có manual PHE mask.
- Không dùng `Seg-CQ500` và `Instance2022` trong training/evaluation bản này.
- Target là binary segmentation: `background` và `PHE`.
- Dùng split mới `99/10/11` từ `02_10` để chạy theo hướng train-heavy hiện tại.

Lưu ý: notebook này đã bật các flag chạy thật. Khi Run All, nó sẽ convert data -> plan/preprocess -> ghi split -> train -> predict -> evaluate.

In [ ]:
from pathlib import Path
import os
import sys
import json
import shutil
import time

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
NNUNET_REPO = PROJECT_ROOT / 'nnUNet-master'
PHE_ROOT = PROJECT_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT'
PHE_IMAGE_DIR = PHE_ROOT / 'set'
PHE_MASK_DIR = PHE_ROOT / 'label'

SPLIT_CSV = PROJECT_ROOT / 'outputs_02_10_pese_guided_3dff_refined_pseudo_iph_phe_25d_segmentation' / 'manifests' / '3dff_iph_phe_patient_split.csv'

OUTPUT_ROOT = PROJECT_ROOT / 'outputs_02_12_nnunetv2_phe_sich_newsplit_baseline'
NNUNET_BASE = OUTPUT_ROOT / 'nnunet_workdir'
NNUNET_RAW = NNUNET_BASE / 'nnUNet_raw'
NNUNET_PREPROCESSED = NNUNET_BASE / 'nnUNet_preprocessed'
NNUNET_RESULTS = NNUNET_BASE / 'nnUNet_results'

DATASET_ID = 112
DATASET_NAME = f'Dataset{DATASET_ID:03d}_PHESICH_PHE_NewSplit'
DATASET_DIR = NNUNET_RAW / DATASET_NAME

for p in [OUTPUT_ROOT, NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw'] = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED)
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)

# Windows/Jupyter safe defaults. nnU-Net background augmentation workers often crash on Windows notebooks.
os.environ['nnUNet_n_proc_DA'] = '0'
os.environ['nnUNet_compile'] = 'False'

if str(NNUNET_REPO) not in sys.path:
    sys.path.insert(0, str(NNUNET_REPO))

assert NNUNET_REPO.exists(), f'Missing nnUNet repo: {NNUNET_REPO}'
assert PHE_IMAGE_DIR.exists(), f'Missing PHE image dir: {PHE_IMAGE_DIR}'
assert PHE_MASK_DIR.exists(), f'Missing PHE mask dir: {PHE_MASK_DIR}'
assert SPLIT_CSV.exists(), f'Missing split CSV from 02_10: {SPLIT_CSV}'

print('Project root          :', PROJECT_ROOT)
print('nnU-Net repo          :', NNUNET_REPO)
print('nnUNet_raw           :', os.environ['nnUNet_raw'])
print('nnUNet_preprocessed  :', os.environ['nnUNet_preprocessed'])
print('nnUNet_results       :', os.environ['nnUNet_results'])
print('nnUNet_n_proc_DA     :', os.environ['nnUNet_n_proc_DA'])
print('nnUNet_compile       :', os.environ['nnUNet_compile'])
print('Dataset              :', DATASET_NAME)
print('Split CSV            :', SPLIT_CSV)


## Optional install

Cell này kiểm tra dependency cần cho nnU-Net. Nếu thiếu `SimpleITK` hoặc dependency chính của nnU-Net, notebook sẽ tự cài bằng `pip` khi `AUTO_INSTALL_MISSING_DEPS = True`.

In [ ]:
INSTALL_NNUNET = False
AUTO_INSTALL_MISSING_DEPS = True

def ensure_import(import_name, pip_name=None):
    import importlib
    import subprocess
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        if not AUTO_INSTALL_MISSING_DEPS:
            raise
        cmd = [sys.executable, '-m', 'pip', 'install', pip_name]
        print('Missing', import_name, '-> running:', ' '.join(cmd))
        subprocess.check_call(cmd)
        return importlib.import_module(import_name)

if INSTALL_NNUNET:
    import subprocess
    cmd = [sys.executable, '-m', 'pip', 'install', '-e', str(NNUNET_REPO)]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

# nnU-Net v2 dependencies used by planning/training/I-O. Keep this explicit so local runs fail early.
ensure_import('SimpleITK')
ensure_import('nibabel')
ensure_import('batchgenerators')
ensure_import('batchgeneratorsv2')
ensure_import('acvl_utils', 'acvl-utils')
ensure_import('dynamic_network_architectures', 'dynamic-network-architectures')
ensure_import('blosc2')
ensure_import('yacs')

try:
    import nnunetv2
    print('nnunetv2 import OK:', nnunetv2.__file__)
except Exception as exc:
    print('nnunetv2 import failed. Set INSTALL_NNUNET=True or install dependencies first.')
    raise exc


## Về nhãn ICH/IPH trong bản nnU-Net này

Bản `02_12` hiện **không đưa ICH/IPH vào label**. Lý do là PHE-SICH trong pipeline hiện tại chỉ có manual PHE mask; ICH/IPH của các notebook 3DFF là pseudo-label sinh từ HU/Otsu/ROI quanh PHE.

Với nnU-Net, nếu train multi-class thì label volume phải có integer class rõ ràng, ví dụ:

```text
0 = background
1 = ICH/IPH
2 = PHE
```

Nếu đưa pseudo ICH nhiễu vào ngay, nnU-Net full 3D có thể học rất mạnh theo nhãn sai và làm kết quả PHE khó phân tích. Vì vậy bản baseline đầu tiên nên là PHE-only để trả lời câu hỏi: full 3D nnU-Net trên manual PHE mask đạt được bao nhiêu.

Sau khi có baseline PHE-only, có thể tạo bản sau `02_12b` theo một trong hai hướng:

1. Multi-class pseudo ICH + PHE: dùng lại rule pseudo ICH của `02_10`, tạo label `0/1/2` cho nnU-Net.
2. ICH teacher: train ICH model từ Seg-CQ500/Instance2022, predict ICH trên PHE-SICH, rồi dùng prediction đó làm nhãn phụ sạch hơn HU-rule.

Hướng 2 hợp lý hơn nếu mục tiêu là cải thiện thật sự, vì ICH pseudo hiện tại vẫn là nguồn nhiễu.

## 1. Inspect split và dữ liệu

Split này lấy từ `02_10`: train 99, val 10, test 11. nnU-Net sẽ train trên `train+val` folder nhưng fold 0 được ép dùng đúng train/val mới bằng `splits_final.json`. Test set được đặt riêng trong `imagesTs/labelsTs`.

In [ ]:
split_df = pd.read_csv(SPLIT_CSV, dtype={'scan_id': str, 'patient_id': str})
split_df.columns = [str(c).strip() for c in split_df.columns]

def make_nnunet_case_id(value, img_path=None):
    value = '' if value is None or pd.isna(value) else str(value).strip()
    if not value and img_path is not None:
        name = Path(str(img_path)).name
        value = name.replace('.nii.gz', '').replace('.nii', '')
    if value.lower().startswith('phe_'):
        return value
    if value.isdigit():
        return f'phe_{int(value):04d}'
    safe = ''.join(ch if ch.isalnum() else '_' for ch in value).strip('_')
    return f'phe_{safe}' if not safe.lower().startswith('phe_') else safe

if 'patient_id' in split_df.columns:
    split_df['case_id'] = [make_nnunet_case_id(v, p) for v, p in zip(split_df['patient_id'], split_df['img_path'])]
elif 'scan_id' in split_df.columns:
    split_df['case_id'] = [make_nnunet_case_id(v, p) for v, p in zip(split_df['scan_id'], split_df['img_path'])]
else:
    split_df['case_id'] = [make_nnunet_case_id(None, p) for p in split_df['img_path']]

required_cols = {'case_id', 'split', 'img_path', 'phe_mask_path'}
missing_cols = required_cols - set(split_df.columns)
assert not missing_cols, f'Missing columns in split CSV: {missing_cols}'

display(split_df.groupby('split').agg(
    cases=('case_id', 'count'),
    slices=('n_slices', 'sum'),
    phe_positive_slices=('phe_positive_slices', 'sum'),
    total_phe_ml=('phe_volume_ml', 'sum'),
))

print('Total cases:', len(split_df))
print('All image paths exist:', split_df['img_path'].map(lambda x: Path(x).exists()).all())
print('All mask paths exist :', split_df['phe_mask_path'].map(lambda x: Path(x).exists()).all())


## 2. Convert PHE-SICH sang nnU-Net v2 raw format

nnU-Net cần cấu trúc:

```text
nnUNet_raw/Dataset112_PHESICH_PHE_NewSplit/
  dataset.json
  imagesTr/phe_0001_0000.nii.gz
  labelsTr/phe_0001.nii.gz
  imagesTs/phe_0003_0000.nii.gz
  labelsTs/phe_0003.nii.gz
```

Label được ép về binary: `0 background`, `1 PHE`.

In [ ]:
OVERWRITE_RAW_DATASET = True

def write_binary_label(src_path: Path, dst_path: Path):
    try:
        import SimpleITK as sitk
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError('Missing SimpleITK. Rerun the Optional install/dependency cell above, or run: pip install SimpleITK') from exc
    src_path = Path(src_path)
    dst_path = Path(dst_path)
    seg = sitk.ReadImage(str(src_path))
    arr = sitk.GetArrayFromImage(seg)
    out = sitk.GetImageFromArray((arr > 0).astype(np.uint8))
    out.CopyInformation(seg)
    sitk.WriteImage(out, str(dst_path))

if DATASET_DIR.exists() and OVERWRITE_RAW_DATASET:
    shutil.rmtree(DATASET_DIR)

imagesTr = DATASET_DIR / 'imagesTr'
labelsTr = DATASET_DIR / 'labelsTr'
imagesTs = DATASET_DIR / 'imagesTs'
labelsTs = DATASET_DIR / 'labelsTs'
for p in [imagesTr, labelsTr, imagesTs, labelsTs]:
    p.mkdir(parents=True, exist_ok=True)

records = []
for row in split_df.itertuples(index=False):
    case_id = str(row.case_id)
    img_src = Path(row.img_path)
    seg_src = Path(row.phe_mask_path)
    if row.split == 'test':
        img_dst = imagesTs / f'{case_id}_0000.nii.gz'
        seg_dst = labelsTs / f'{case_id}.nii.gz'
    else:
        img_dst = imagesTr / f'{case_id}_0000.nii.gz'
        seg_dst = labelsTr / f'{case_id}.nii.gz'
    if OVERWRITE_RAW_DATASET or not img_dst.exists():
        shutil.copy2(img_src, img_dst)
    if OVERWRITE_RAW_DATASET or not seg_dst.exists():
        write_binary_label(seg_src, seg_dst)
    records.append({
        'case_id': case_id,
        'split': row.split,
        'nnunet_image': str(img_dst),
        'nnunet_label': str(seg_dst),
    })

dataset_json = {
    'channel_names': {'0': 'CT'},
    'labels': {'background': 0, 'PHE': 1},
    'numTraining': int((split_df['split'] != 'test').sum()),
    'file_ending': '.nii.gz',
    'overwrite_image_reader_writer': 'SimpleITKIO',
}
with open(DATASET_DIR / 'dataset.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_json, f, indent=2)

conversion_df = pd.DataFrame(records)
conversion_df.to_csv(OUTPUT_ROOT / '02_12_nnunet_conversion_manifest.csv', index=False)

print('Dataset dir:', DATASET_DIR)
print('imagesTr:', len(list(imagesTr.glob('*.nii.gz'))))
print('labelsTr:', len(list(labelsTr.glob('*.nii.gz'))))
print('imagesTs:', len(list(imagesTs.glob('*.nii.gz'))))
print('labelsTs:', len(list(labelsTs.glob('*.nii.gz'))))
print('dataset.json:', DATASET_DIR / 'dataset.json')
display(conversion_df.groupby('split').size().rename('cases').reset_index())


## 3. Helper gọi entrypoint của nnU-Net

Các command `nnUNetv2_*` thường chạy từ terminal. Trong notebook này gọi trực tiếp entrypoint Python để không phụ thuộc PATH của Windows.

In [ ]:
def call_entrypoint(entrypoint_func, argv):
    old_argv = sys.argv[:]
    sys.argv = list(argv)
    print('>>>', ' '.join(map(str, sys.argv)))
    t0 = time.time()
    try:
        return entrypoint_func()
    finally:
        sys.argv = old_argv
        print(f'Done in {(time.time() - t0) / 60:.1f} min')


## 4. Plan and preprocess

`RUN_PLAN_PREPROCESS` đang bật sẵn để chạy thật. Bản này mặc định preprocess `3d_fullres`. Nếu GPU/RAM yếu, đổi `CONFIGURATION = '2d'`.

In [ ]:
CONFIGURATION = '3d_fullres'  # alternatives: '2d', '3d_fullres'
RUN_PLAN_PREPROCESS = True

if RUN_PLAN_PREPROCESS:
    from nnunetv2.experiment_planning.plan_and_preprocess_entrypoints import plan_and_preprocess_entry
    # Planner calls torch.set_num_threads(get_allowed_n_proc_DA()), so this must be >= 1.
    # Training cell below will switch it back to 0 to avoid Windows/Jupyter background-worker crashes.
    old_n_proc_da = os.environ.get('nnUNet_n_proc_DA')
    os.environ['nnUNet_n_proc_DA'] = '1'
    try:
        call_entrypoint(plan_and_preprocess_entry, [
            'nnUNetv2_plan_and_preprocess',
            '-d', str(DATASET_ID),
            '-c', CONFIGURATION,
            '--verify_dataset_integrity',
            '-npfp', '2',
            '-np', '1',
        ])
    finally:
        os.environ['nnUNet_n_proc_DA'] = old_n_proc_da if old_n_proc_da is not None else '0'
else:
    print('Skipped. Set RUN_PLAN_PREPROCESS=True to run nnU-Net planning/preprocessing.')


## 5. Ghi manual split cho fold 0

Sau khi plan/preprocess, chạy cell này để ép fold 0 dùng đúng split mới:

- train: 99 case
- val: 10 case
- test: 11 case, nằm ngoài training và chỉ dùng predict/evaluate

In [ ]:
preprocessed_dataset_dir = NNUNET_PREPROCESSED / DATASET_NAME
split_json_path = preprocessed_dataset_dir / 'splits_final.json'

train_ids = split_df.loc[split_df['split'] == 'train', 'case_id'].astype(str).tolist()
val_ids = split_df.loc[split_df['split'] == 'val', 'case_id'].astype(str).tolist()
test_ids = split_df.loc[split_df['split'] == 'test', 'case_id'].astype(str).tolist()

manual_splits = [{'train': train_ids, 'val': val_ids}]

if preprocessed_dataset_dir.exists():
    with open(split_json_path, 'w', encoding='utf-8') as f:
        json.dump(manual_splits, f, indent=2)
    print('Wrote:', split_json_path)
    print('train:', len(train_ids), 'val:', len(val_ids), 'test:', len(test_ids))
else:
    print('Preprocessed folder not found yet:', preprocessed_dataset_dir)
    print('Run plan/preprocess first, then rerun this cell.')


## 6. Train nnU-Net fold 0

Các lựa chọn trainer:

- `nnUNetTrainer_5epochs`: smoke test nhanh, chỉ để kiểm tra pipeline.
- `nnUNetTrainer_250epochs`: bản nhẹ hơn default, hợp lý để thử trước.
- `nnUNetTrainer`: default nnU-Net 1000 epochs, nặng nhất nhưng đúng baseline chuẩn hơn.

`RUN_TRAIN` đang bật sẵn. Mặc định dùng `nnUNetTrainer_250epochs`: đây là run thật, không phải smoke test 5 epoch, nhưng vẫn nhẹ hơn default `nnUNetTrainer` 1000 epoch.

In [ ]:
RUN_TRAIN = True
FOLD = 0
TRAINER = 'nnUNetTrainer_250epochs'
PLANS = 'nnUNetPlans'
EXPORT_VAL_NPZ = True
CONTINUE_TRAINING = True

if RUN_TRAIN:
    import torch
    from nnunetv2.run.run_training import run_training
    os.environ['nnUNet_n_proc_DA'] = '0'
    os.environ['nnUNet_compile'] = 'False'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Training device:', device)
    print('nnUNet_n_proc_DA:', os.environ.get('nnUNet_n_proc_DA'))
    print('nnUNet_compile:', os.environ.get('nnUNet_compile'))
    run_training(
        str(DATASET_ID),
        CONFIGURATION,
        str(FOLD),
        trainer_class_name=TRAINER,
        plans_identifier=PLANS,
        export_validation_probabilities=EXPORT_VAL_NPZ,
        continue_training=CONTINUE_TRAINING,
        device=device,
    )
else:
    print('Skipped. Set RUN_TRAIN=True to train.')


## 7. Predict test set và evaluate

Cell này predict 11 test case trong `imagesTs` và so với `labelsTs`. Nếu dùng trainer/config khác lúc train, phải giữ cùng `TRAINER`, `CONFIGURATION`, `PLANS` ở đây.

In [25]:
RUN_PREDICT = True
RUN_EVALUATE = True
DISABLE_TTA = False
CHECKPOINT = 'auto'  # auto: best -> final -> latest; or set checkpoint_best.pth/checkpoint_final.pth manually

MODEL_DIR = NNUNET_RESULTS / DATASET_NAME / f'{TRAINER}__{PLANS}__{CONFIGURATION}'
FOLD_DIR = MODEL_DIR / f'fold_{FOLD}'

def resolve_checkpoint_name(checkpoint_setting='auto'):
    if checkpoint_setting != 'auto':
        ckpt = FOLD_DIR / checkpoint_setting
        if not ckpt.exists():
            available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
            raise FileNotFoundError(f'Missing checkpoint: {ckpt}. Available: {available}')
        return checkpoint_setting
    for name in ['checkpoint_best.pth', 'checkpoint_final.pth', 'checkpoint_latest.pth']:
        if (FOLD_DIR / name).exists():
            return name
    available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
    raise FileNotFoundError(f'No checkpoint found in {FOLD_DIR}. Available: {available}')

RESOLVED_CHECKPOINT = resolve_checkpoint_name(CHECKPOINT)
print('Model dir          :', MODEL_DIR)
print('Fold dir           :', FOLD_DIR)
print('Using checkpoint   :', RESOLVED_CHECKPOINT)

PRED_DIR = OUTPUT_ROOT / f'predictions_{CONFIGURATION}_{TRAINER}_fold{FOLD}_{RESOLVED_CHECKPOINT.replace(".pth", "")}'
SUMMARY_JSON = PRED_DIR / 'summary.json'

if RUN_PREDICT:
    import gc
    import torch
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

    # Do not call nnUNetv2_predict entrypoint inside a post-training notebook kernel.
    # The entrypoint calls torch.set_num_interop_threads(1), which fails after PyTorch parallel work has started.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Predict device:', device)

    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=not DISABLE_TTA,
        perform_everything_on_device=(device.type == 'cuda'),
        device=device,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=True,
    )
    predictor.initialize_from_trained_model_folder(
        str(MODEL_DIR),
        use_folds=(FOLD,),
        checkpoint_name=RESOLVED_CHECKPOINT,
    )
    predictor.predict_from_files(
        str(imagesTs),
        str(PRED_DIR),
        save_probabilities=False,
        overwrite=True,
        num_processes_preprocessing=1,
        num_processes_segmentation_export=1,
        folder_with_segs_from_prev_stage=None,
        num_parts=1,
        part_id=0,
    )
else:
    print('Skipped prediction. Set RUN_PREDICT=True to run inference.')

if RUN_EVALUATE:
    from nnunetv2.evaluation.evaluate_predictions import evaluate_folder_entry_point
    plans_file = NNUNET_PREPROCESSED / DATASET_NAME / f'{PLANS}.json'
    dataset_json_file = DATASET_DIR / 'dataset.json'
    call_entrypoint(evaluate_folder_entry_point, [
        'nnUNetv2_evaluate_folder',
        str(labelsTs),
        str(PRED_DIR),
        '-djfile', str(dataset_json_file),
        '-pfile', str(plans_file),
        '-o', str(SUMMARY_JSON),
        '-np', '2',
    ])
else:
    print('Skipped evaluation. Set RUN_EVALUATE=True after prediction.')

print('Prediction dir:', PRED_DIR)
print('Summary json  :', SUMMARY_JSON)


Model dir          : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_12_nnunetv2_phe_sich_newsplit_baseline\nnunet_workdir\nnUNet_results\Dataset112_PHESICH_PHE_NewSplit\nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres
Fold dir           : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_12_nnunetv2_phe_sich_newsplit_baseline\nnunet_workdir\nnUNet_results\Dataset112_PHESICH_PHE_NewSplit\nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres\fold_0
Using checkpoint   : checkpoint_best.pth
Predict device: cuda
There are 11 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 11 cases that I would like to predict

Predicting phe_0002:
perform_everything_on_device: True


100%|██████████| 27/27 [00:14<00:00,  1.84it/s]


sending off prediction to background worker for resampling and export
done with phe_0002

Predicting phe_0029:
perform_everything_on_device: True


100%|██████████| 12/12 [00:05<00:00,  2.14it/s]


sending off prediction to background worker for resampling and export
done with phe_0029

Predicting phe_0033:
perform_everything_on_device: True


...

100%|██████████| 27/27 [00:13<00:00,  2.00it/s]


sending off prediction to background worker for resampling and export
done with phe_0116

Predicting phe_0167:
perform_everything_on_device: True


100%|██████████| 12/12 [00:05<00:00,  2.14it/s]


sending off prediction to background worker for resampling and export
done with phe_0167
GPU prediction completed. Waiting for remaining segmentation exports to finish...


Segmentation export complete.
>>> nnUNetv2_evaluate_folder D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_12_nnunetv2_phe_sich_newsplit_baseline\nnunet_workdir\nnUNet_raw\Dataset112_PHESICH_PHE_NewSplit\labelsTs D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_12_nnunetv2_phe_sich_newsplit_baseline\predictions_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best -djfile D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_12_nnunetv2_phe_sich_newsplit_baseline\nnunet_workdir\nnUNet_raw\Dataset112_PHESICH_PHE_NewSplit\dataset.json -pfile D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_12_nnunetv2_phe_sich_newsplit_baseline\nnunet_workdir\nnUNet_preprocessed\Dataset112_PHESICH_PHE_NewSplit\nnUNetPlans.json -o D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_12_nnunetv2_phe_sich_newsplit_baseline\predictions_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best\summary.json -np 2
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer
Done in 0.0 min
Prediction dir: D:\Thuy_L

## 8. Đọc kết quả summary

nnU-Net summary mặc định có Dice và IoU. Các metric như HD95/ASSD/NSD/RVD có thể bổ sung sau bằng evaluator đang dùng ở các notebook 3DFF.

In [26]:
if SUMMARY_JSON.exists():
    with open(SUMMARY_JSON, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    print('Foreground mean:')
    print(json.dumps(summary.get('foreground_mean', {}), indent=2))
    print('\nPHE label metrics:')
    print(json.dumps(summary.get('mean', {}).get('1', {}), indent=2))
else:
    print('No summary yet:', SUMMARY_JSON)
    print('Run train -> predict -> evaluate first.')


Foreground mean:
{
  "Dice": 0.394140783709968,
  "FN": 1127.1818181818182,
  "FP": 1455.1818181818182,
  "IoU": 0.27095590081275495,
  "TN": 7836581.545454546,
  "TP": 1324.8181818181818,
  "n_pred": 2780.0,
  "n_ref": 2452.0
}

PHE label metrics:
{
  "Dice": 0.394140783709968,
  "FN": 1127.1818181818182,
  "FP": 1455.1818181818182,
  "IoU": 0.27095590081275495,
  "TN": 7836581.545454546,
  "TP": 1324.8181818181818,
  "n_pred": 2780.0,
  "n_ref": 2452.0
}


## Ghi chú so sánh với 3DFF-Net

Bản `02_12` là PHE-only nnU-Net baseline. Nó không dùng pseudo IPH và không dùng PESE-guided fusion. Nếu kết quả tốt hơn 3DFF, lý do chính có thể là nnU-Net dùng full 3D U-Net và tự động chọn preprocessing/patch/batch theo dataset. Nếu kết quả không tốt hơn, cần kiểm tra lại split, preprocessing, số epoch, checkpoint và metric protocol.

Bước tiếp theo nếu bản này chạy ổn:

1. Chạy `nnUNetTrainer_250epochs` để test nhanh hơn default.
2. Nếu Dice có triển vọng, chạy `nnUNetTrainer` default 1000 epochs.
3. Bổ sung evaluator HD95/ASSD/NSD/RVD giống `02_10b` để so đầy đủ.
4. Sau khi có baseline PHE-only, mới cân nhắc dùng Seg-CQ500/Instance2022 để pretrain nhánh ICH/IPH.